In [ ]:
'''
Work in progress for tetrad.example

I may return to this to move the data handling into utils.py
Still have much to do for tetrad
Primarily I am having issues with needing Java on the Docker image -- hoping to get to office hours about this
'''


# Tetrad Example

This script will be an example of how to use Tetrad to analyze causal relationships with regard to user posts taken from Twitter and associated stock market data over time.  

In [1]:
!sudo pip install -U --quiet datasets
!sudo pip install -U --quiet JPype1
!sudo pip install -U --quiet git+https://github.com/cmu-phil/py-tetrad

## Import Packages

In [2]:
import pandas as pd
from datetime import datetime

## Import and clean tweet sentiment data
Using the huggingface dataset emad12/stock_tweets_sentiment

In [3]:
splits = {'train': 'data/train-00000-of-00001-49baa0648effea14.parquet', 'test': 'data/test-00000-of-00001-cb0233e05c1cc1c9.parquet'}
twt_df = pd.read_parquet("hf://datasets/emad12/stock_tweets_sentiment/" + splits["train"])

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
tweet_df = twt_df.copy(deep=True)
def fix_dates(r):
    first_str = r['post_date'].split(None, 1)[0]
    if '-' in first_str:
        return first_str
    elif first_str.isdigit():
        return datetime.fromtimestamp(int(first_str))
    else:
        return datetime.strptime(r['post_date'], '%a %b %d %X %z %Y').strftime('%Y-%m-%d')
tweet_df['post_date'] = tweet_df.apply(fix_dates, axis=1)
tweet_df['post_date'] = pd.to_datetime(tweet_df['post_date']).dt.strftime('%Y-%m-%d')
tweet_df = tweet_df[tweet_df['ticker_symbol'].isin(['AMZN', 'AAPL', 'TSLA', 'GOOG', 'MSFT'])]

In [7]:
print(tweet_df['ticker_symbol'].value_counts())
print(tweet_df['post_date'].min())
print(tweet_df['post_date'].max())

ticker_symbol
TSLA    34440
AAPL    21281
AMZN    10026
GOOG     6132
MSFT     2563
Name: count, dtype: int64
2015-01-01
2019-12-31


## Import and clean historical stock data for selected stocks
Using the huggingface dataset no-ry/world-stock-prices-daily-updating

In [8]:
st_df = pd.read_csv("hf://datasets/no-ry/world-stock-prices-daily-updating/World-Stock-Prices-Dataset.csv")

In [9]:
stock_df = st_df.copy(deep=True)
stock_df['Date'] = pd.to_datetime(stock_df['Date'], utc=True).dt.strftime('%Y-%m-%d')
stock_df = stock_df.drop(columns=['Industry_Tag', 'Country', 'Dividends', 'Stock Splits', 'Capital Gains', 'Brand_Name'])
stock_df = stock_df[stock_df['Ticker'].isin(['AMZN', 'AAPL', 'TSLA', 'GOOG', 'MSFT'])]
stock_df = stock_df.drop_duplicates(subset=['Date', 'Ticker'], keep='last')
stock_df['Delta'] = stock_df['Close'] - stock_df['Open']


In [10]:
stock_df.head(5)

,Date,Open,High,Low,Close,Volume,Ticker,Delta
86,2025-07-03,212.149994,214.649994,211.809998,213.550003,34955800.0,AAPL,1.400009
96,2025-07-03,493.809998,500.130005,493.440002,498.839996,13984800.0,MSFT,5.029999
97,2025-07-03,221.820007,224.009995,221.360001,223.410004,29632400.0,AMZN,1.589996
99,2025-07-03,317.989990,318.450012,312.760010,315.350006,58042300.0,TSLA,-2.639984
111,2025-07-02,219.729996,221.600006,219.059998,219.919998,30840800.0,AMZN,0.190002


In [12]:
#Need further investigation to load java onto the docker image before I can actually run Tetrad here

import pandas as pd
#import jpype
#import jpype.imports
#import pytetrad.tools.TetradSearch as ts

#jpype.startJVM(classpath=[f"resources/tetrad-gui-current-launch.jar"])

JVMNotFoundException: No JVM shared library file (libjvm.so) found. Try setting up the JAVA_HOME environment variable properly.